In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

plik_swir = 'LC_clipped_band6.tif' 
plik_nir = 'LC_clipped_band5.tif'

with rasterio.open(plik_swir) as src_swir:
    swir = src_swir.read(1).astype('float32')
    transform = src_swir.transform
    crs = src_swir.crs
    width = src_swir.width
    height = src_swir.height

with rasterio.open(plik_nir) as src_nir:
    nir = src_nir.read(1).astype('float32')

np.seterr(divide='ignore', invalid='ignore')

maska_tla = (swir == 0) | np.isnan(swir) | (nir == 0) | np.isnan(nir)

ndbi = np.zeros_like(swir)
piksele_danych = ~maska_tla
ndbi[piksele_danych] = (swir[piksele_danych] - nir[piksele_danych]) / (swir[piksele_danych] + nir[piksele_danych])

vmin = -0.2
vmax = 0.4

ndbi_norm = (ndbi - vmin) / (vmax - vmin)
ndbi_norm = np.clip(ndbi_norm, 0, 1)

kolory_ndbi = ['#e0e0e0', '#ffaa00', '#e60000', '#660000']
cmap = LinearSegmentedColormap.from_list('ndbi_palette', kolory_ndbi)

colored_image = cmap(ndbi_norm) 


colored_image[maska_tla, 3] = 0.0 

ndbi_rgba = (colored_image * 255).astype('uint8')
ndbi_rgba_chw = np.transpose(ndbi_rgba, (2, 0, 1)) 

profil = {
    'driver': 'GTiff',
    'height': height,
    'width': width,
    'count': 4, 
    'dtype': 'uint8',
    'crs': crs,
    'transform': transform,
    'compress': 'lzw'
}

with rasterio.open('Krakow_NDBI_RGB.tif', 'w', **profil) as dst:
    dst.write(ndbi_rgba_chw)

print("Gotowe")

Sukces! Wygenerowano poprawiony, przezroczysty plik z wyraźnymi kolorami betonu.
